# FABN Bond Portfolio Optimizer — Gurobi

**Goal:** Maximize the Net Economic Value (NEV) of a Funding Agreement-Backed Note (FABN)
strategy by optimally allocating capital $h_i$ across a fixed-income universe.

---

## Objective Function

$$
\max_{h} \left[
\underbrace{\sum_{i \in \mathcal{S}} \bigl(Spread_i + \beta\, Signal_i\bigr)\, h_i}_{\text{Relative Value}}
\;-\;
\gamma \underbrace{\bigl(C_1(h) + C_3(h)\bigr)}_{\text{Capital cost}}
\;-\;
\underbrace{\sum_{i} \tau_i\, |h_i - h_i^{curr}|}_{\text{Transaction cost}}
\;-\;
\lambda\, \underbrace{CFShortfall(h)}_{\text{CF shortfall penalty}}
\right]
$$

| Component | Formula | Notes |
|---|---|---|
| $Spread_i$ | $y_i - r^{FABN}$ | Yield minus funding rate |
| $C_1(h)$ | $\sum_i \theta_i h_i$ | C-1 credit risk capital |
| $C_3(h)$ | $\alpha \left\lvert \sum_i D_i h_i - D^{FABN} H \right\rvert$ | C-3 ALM mismatch; $H = \sum_i h_i$ fixed by budget |
| $CFShortfall(h)$ | $\sum_t \max(0,\, CF_t^L - CF_t^A(h))$ | One-sided CF penalty |

---

## Linearization

All nonlinear terms are reformulated using auxiliary variables and linear constraints, resulting in a standard linear program (LP) solvable by Gurobi.

- Absolute Value: |x| → d⁺ + d⁻
- Max: max(0, x) → s ≥ x, s ≥ 0

| Nonlinear term | Auxiliary vars | Reformulation |
|---|---|---|
| $C_3$: absolute value of duration gap | $d^+, d^- \geq 0$ | $d^+ - d^- = \sum D_i h_i - D^{FABN}H$; $C_3 = \alpha(d^+ + d^-)$ |
| Transaction costs: $\|h_i - h_i^{curr}\|$ | $tc^+_i,\, tc^-_i \geq 0$ | $tc^+_i - tc^-_i = h_i - h_i^{curr}$; cost $= \tau_i(tc^+_i + tc^-_i)$ |
| CF shortfall: $\max(0, CF^L_t - CF^A_t)$ | $s_t \geq 0$ | $s_t \geq CF^L_t - CF^A_t$ |
| Duration constraint: $\|D_{avg} - D^{FABN}\| \leq \varepsilon_D$ | same $d^+, d^-$ | $d^+ \leq \varepsilon_D H$;  $d^- \leq \varepsilon_D H$ |

---

## Pipeline Overview
0. **Imports** — libraries
1. **Data & Parameters** — load inputs
2. **Gurobi Model** — formulate and solve the LP
3. **Results** — extract allocations and NEV breakdown
4. **Export** — generate report

---
## Section 0 — Imports

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
from google.cloud import bigquery
import matplotlib.pyplot as plt

PROJECT_ID = "insurance-backed-securities"
DATASET    = "Securities"

client = bigquery.Client(project=PROJECT_ID)
print(f"Connected to project: {client.project}")

---
## Section 1 — Data & Parameters

In [ ]:
# =============================================================================
# 1A. DATA LOADING
# =============================================================================

# --- Bond universe ---
# Fill and process with GCP data
yields = # yields
durs   = # Durations
theta  = # C-1 charge
tau    = # transaction cost
signal = # composite signal
h_curr = # current (equal-weight) allocations
spread = yields - r_FABN # Spread_i = y_i - r^FABN
score = spread + beta_w * signal # Score_i = Spread_i + β * Signal_i

# --- FABN / liability parameters ---
# Fill with Athene generated information
H       = # total capital to invest — budget H ($)
r_FABN  = # funding agreement crediting rate (annual, exogenous)
D_FABN  = # liability modified duration (years)
C_curr  = # current regulatory capital ($)
C_min   = # minimum required capital — RBC floor ($)
RBC_bar = # minimum RBC solvency ratio
dt      = # time scaling factor (annualized)

# --- Cash flow matrices ---
# Complete
bond_cf = # (T x N) Matrix
fabn_cf = # (T,) Matrix
# FABN Payments
qtr_bond_cf  = # (Q, N) Matrix
qtr_fabn_cf  = # (Q,) Matrix
qtr_idx = # payment periods


# =============================================================================
# 1B. MODEL DIMENSIONS
# =============================================================================

N = # number of bonds in the universe
T = # horizon
Q = # periods where FABN pays

# =============================================================================
# 1C. OBJECTIVE PENALTY WEIGHTS
# =============================================================================

gamma_w  =  # γ : weight on total capital cost (C1 + C3)
beta_w   =  # β : weight on market signal in score
alpha_w  =  # α : scaling factor for C3 duration mismatch
lambda_w =  # λ : weight on cash flow shortfall penalty

# =============================================================================
# 1D. CONSTRAINT PARAMETERS
# =============================================================================

eps_D =     # duration tolerance band: |D_avg - D_FABN| ≤ eps_D

---
## Section 2 — Gurobi Optimization Model

In [ ]:
# =============================================================================
# 2. GUROBI OPTIMIZATION MODEL
# =============================================================================

model = gp.Model("FABN_NEV_Optimizer")
model.Params.LogToConsole = 1   # set to 0 to suppress solver output

# ---------------------------------------------------------------------------
# 2A. DECISION VARIABLES
# ---------------------------------------------------------------------------

# h[i] ≥ 0 : capital allocated to bond i ($)
# Lower bound lb=0 enforces the long-only constraint implicitly.
h = model.addVars(N, lb=0.0, name="h")

# ---------------------------------------------------------------------------
# 2B. AUXILIARY VARIABLES  (linearization of nonlinear terms)
# ---------------------------------------------------------------------------

# --- C3 duration-gap linearization ---
# Since Σ h_i = H (budget), D_avg = Σ D_i h_i / H, so:
#   C3 = α |Σ D_i h_i - D_FABN * H|
# Introduce d_pos, d_neg ≥ 0 such that:
#   d_pos - d_neg = Σ D_i h_i - D_FABN * H
#   C3 = α * (d_pos + d_neg)
d_pos = model.addVar(lb=0.0, name="d_pos")   # positive part of duration gap
d_neg = model.addVar(lb=0.0, name="d_neg")   # negative part of duration gap

# --- Transaction cost linearization ---
# |h_i - h_curr_i| = tc_plus_i + tc_minus_i
# h_i - h_curr_i   = tc_plus_i - tc_minus_i  (decomposition into positive/negative parts)
tc_plus  = model.addVars(N, lb=0.0, name="tc_plus")
tc_minus = model.addVars(N, lb=0.0, name="tc_minus")

# --- Cash flow shortfall linearization ---
# s[q] = max(0, CF_L_q - CF_A_q) at each quarterly period q
# Enforced via:  s[q] ≥ CF_L_q - CF_A_q  and  s[q] ≥ 0 (lb)
s = model.addVars(Q, lb=0.0, name="s")

# ---------------------------------------------------------------------------
# 2C. OBJECTIVE FUNCTION
# ---------------------------------------------------------------------------

# (1) Spread income:  Σ_i Score_i * h_i   where Score_i = Spread_i + β * Signal_i
spread_income = gp.quicksum(score[i] * h[i] for i in range(N))

# (2) C1 — credit risk capital cost:  Σ_i θ_i * h_i
C1 = gp.quicksum(c1_th[i] * h[i] for i in range(N))

# (3) C3 — ALM duration mismatch cost:  α * (d_pos + d_neg)
C3 = alpha_w * (d_pos + d_neg)

# (4) Total capital cost:  γ * (C1 + C3)
capital_cost = gamma_w * (C1 + C3)

# (5) Transaction costs:  Σ_i τ_i * (tc_plus_i + tc_minus_i)
txn_cost = gp.quicksum(tau[i] * (tc_plus[i] + tc_minus[i]) for i in range(N))

# (6) Cash flow shortfall penalty:  λ * Σ_q s_q
cf_shortfall_penalty = lambda_w * gp.quicksum(s[q] for q in range(Q))

# NEV = (1) - (4) - (5) - (6)
NEV = spread_income - capital_cost - txn_cost - cf_shortfall_penalty
model.setObjective(NEV, GRB.MAXIMIZE)

# ---------------------------------------------------------------------------
# 2D. CORE CONSTRAINTS  (base model — first iteration)
# ---------------------------------------------------------------------------

# C1 — Budget:  Σ h_i = H
model.addConstr(
    gp.quicksum(h[i] for i in range(N)) == H,
    name="budget"
)

# C2 — Solvency / RBC:
#   (C_curr + Σ Spread_i * h_i * dt) / C_min ≥ RBC_bar
#   ↔  Σ Spread_i * h_i  ≥  (RBC_bar * C_min - C_curr) / dt
rbc_rhs = (RBC_bar * C_min - C_curr) / dt
model.addConstr(
    gp.quicksum(spread[i] * h[i] for i in range(N)) >= rbc_rhs,
    name="solvency"
)

# C3 — Duration alignment:  |D_avg - D_FABN| ≤ eps_D
#   Since H is fixed by budget: |Σ D_i h_i - D_FABN * H| ≤ eps_D * H
#   Linearized through d_pos and d_neg:
model.addConstr(
    gp.quicksum(durs[i] * h[i] for i in range(N)) - D_FABN * H == d_pos - d_neg,
    name="dur_gap_decomp"
)
model.addConstr(d_pos <= eps_D * H, name="dur_upper")   # D_avg ≤ D_FABN + eps_D
model.addConstr(d_neg <= eps_D * H, name="dur_lower")   # D_avg ≥ D_FABN - eps_D

# Auxiliary — transaction cost decomposition:  h_i - h_curr_i = tc_plus_i - tc_minus_i
for i in range(N):
    model.addConstr(
        h[i] - h_curr[i] == tc_plus[i] - tc_minus[i],
        name=f"tc_decomp_{i}"
    )

# Auxiliary — cash flow shortfall:  s_q ≥ CF_L_q - CF_A_q  for each quarter q
# Surplus periods (CF_A > CF_L) are not penalized — s_q stays at its lb=0.
for q in range(Q):
    CF_A_q = gp.quicksum(qtr_bond_cf[q, i] * h[i] for i in range(N))
    CF_L_q = float(qtr_fabn_cf[q])
    model.addConstr(s[q] >= CF_L_q - CF_A_q, name=f"cf_shortfall_{q}")

# ---------------------------------------------------------------------------
# 2E. SOLVE
# ---------------------------------------------------------------------------

model.optimize()

---
## Section 3 — Results

In [ ]:
# =============================================================================
# 3. RESULTS EXTRACTION  
# =============================================================================

if model.Status == GRB.OPTIMAL:

    # --- Optimal allocations ---
    h_opt = np.array([h[i].X for i in range(N)])

    # --- NEV component breakdown ---
    nev_val           = model.ObjVal
    spread_income_val = float(sum(score[i] * h_opt[i] for i in range(N)))
    C1_val            = float(sum(c1_th[i] * h_opt[i] for i in range(N)))
    C3_val            = alpha_w * (d_pos.X + d_neg.X)
    capital_cost_val  = gamma_w * (C1_val + C3_val)
    txn_cost_val      = float(sum(tau[i] * (tc_plus[i].X + tc_minus[i].X) for i in range(N)))
    shortfall_total   = float(sum(s[q].X for q in range(Q)))
    cf_penalty_val    = lambda_w * shortfall_total

    # --- Portfolio-level metrics ---
    D_avg   = float(sum(durs[i] * h_opt[i] for i in range(N))) / H
    RBC_val = (C_curr + float(sum(spread[i] * h_opt[i] for i in range(N))) * dt) / C_min

    # --- Print summary ---
    print(f"{'='*55}")
    print(f"  OPTIMAL NEV              : ${nev_val:>12,.2f}")
    print(f"  (1) Spread income        : ${spread_income_val:>12,.2f}")
    print(f"  (4) Capital cost γ(C1+C3): ${capital_cost_val:>12,.2f}")
    print(f"      - C1 (credit risk)   : ${C1_val:>12,.2f}")
    print(f"      - C3 (duration ALM)  : ${C3_val:>12,.2f}")
    print(f"  (5) Transaction costs    : ${txn_cost_val:>12,.2f}")
    print(f"  (6) CF shortfall penalty : ${cf_penalty_val:>12,.2f}")
    print(f"{'='*55}")
    print(f"  Portfolio D_avg          :  {D_avg:.4f} yrs  (target {D_FABN} ± {eps_D})")
    print(f"  RBC ratio                :  {RBC_val:.2f}x  (min {RBC_bar:.1f}x)")
    print(f"{'='*55}")
    print(f"  Allocations:")
    for i in range(N):
        print(f"    {BONDS[i][0]:22s}  h = ${h_opt[i]:>10,.2f}  "
              f"(score={score[i]:+.4f})")

    # --- Constraint status table ---
    # Build a DataFrame with each constraint's value, bound, and pass/fail
    # constraints_df = pd.DataFrame({...})
    # display(constraints_df)

    # --- Quarterly cash flow comparison ---
    CF_A_vals = [sum(qtr_bond_cf[q, i] * h_opt[i] for i in range(N)) for q in range(Q)]
    cf_df = pd.DataFrame({
        "Payment"      : qtr_idx,
        "FABN CF ($)"  : qtr_fabn_cf,
        "Asset CF ($)" : CF_A_vals,
        "Surplus ($)"  : [a - l for a, l in zip(CF_A_vals, qtr_fabn_cf)],
        "Shortfall ($)": [s[q].X for q in range(Q)],
    })
    display(cf_df)

elif model.Status == GRB.INFEASIBLE:
    print("Model is INFEASIBLE. Computing IIS to identify conflicting constraints...")
    # model.computeIIS()
    # model.write("infeasible.ilp")

else:
    print(f"Solver status code: {model.Status} — no optimal solution found.")

---
## Section 4 — Export

In [ ]:
# =============================================================================
# 4. EXPORT & REPORTING
# =============================================================================

# --- Export allocation table to CSV ---
results_df = pd.DataFrame({
    "Bond"         : [b[0] for b in BONDS],
    "h_opt ($)"    : h_opt,
    "h_curr ($)"   : h_curr,
    "Delta ($)"    : h_opt - h_curr,
    "Spread (bps)" : spread * 10_000,
    "Score"        : score,
    "C1 charge ($)": c1_th * h_opt,
})
results_df.to_csv("optimizer_results.csv", index=False)
display(results_df)

# --- NEV breakdown summary ---
summary = pd.Series({
    "NEV ($)"               : nev_val,
    "Spread Income ($)"     : spread_income_val,
    "Capital Cost ($)"      : capital_cost_val,
    "Transaction Costs ($)" : txn_cost_val,
    "CF Shortfall Penalty ($)": cf_penalty_val,
    "D_avg (yrs)"           : D_avg,
    "RBC Ratio (x)"         : RBC_val,
})
summary.to_frame("Value").to_excel("nev_summary.xlsx")

# --- Visualizations ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar([b[0] for b in BONDS], h_opt, color="steelblue")
axes[0].set_title("Optimal Allocation $h_i$")
axes[0].set_ylabel("Capital ($)")
axes[0].tick_params(axis='x', rotation=45)

components = [spread_income_val, -capital_cost_val, -txn_cost_val, -cf_penalty_val]
labels     = ["Spread Income", "Capital Cost", "Txn Cost", "CF Shortfall"]
colors     = ["green" if v >= 0 else "red" for v in components]
axes[1].bar(labels, components, color=colors)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("NEV Decomposition")

plt.tight_layout()
plt.savefig("nev_breakdown.png", dpi=150)
plt.show()

print("Pipeline skeleton complete.")